# 🧠 Battlefield Blueprint — Huấn Luyện AI Bot Trên Google Colab (GPU + Drive Sync)

Notebook này cung cấp quy trình tự động hóa hoàn chỉnh để huấn luyện mạng nơ-ron **Deep Dual-Head MLP** (170,005 tham số) cho AI Bot chiến thuật thời gian thực (RTS) trên nền tảng **Google Colab (GPU NVIDIA Tesla T4 / L4 / A100)**.

### 🌟 Điểm nổi bật kỹ thuật:
1. **Kiến trúc Lai siêu tốc (Hybrid Architecture)**: Mô phỏng vật lý, kinh tế và bản đồ không gian trên CPU với bộ đệm phẳng `SpatialGrid` (Zero Heap Allocation), đồng thời đẩy toàn bộ phép nhân ma trận và lan truyền ngược của mạng nơ-ron xuống **Persistent VRAM Buffer của GPU (CUDA)**.
2. **Cơ chế Lười trích xuất (Lazy State Sampling)**: Loại bỏ 75% các phép tính trích xuất vector trạng thái thừa ở các bước tick trung gian, chỉ tính toán tại chu kỳ ra quyết định 1.0s.
3. **Tự Động Đồng Bộ Google Drive Chống Mất Dữ Liệu**: Trọng số model tự động lưu và sao lưu trực tiếp vào thư mục Google Drive của bạn sau mỗi giai đoạn kịch bản (`1v1`, `2v2`, `3v3`, `4v4`, `2v2v2`, `FFA8`). Dù Colab bị ngắt kết nối giữa chừng, bạn vẫn không bao giờ mất kết quả đã huấn luyện!

## Bước 1: Kiểm Tra GPU Phần Cứng (NVIDIA CUDA)

In [ ]:
# Kiểm tra xem Colab đã cấp phát GPU chưa (Menu: Runtime -> Change runtime type -> T4 GPU / L4 GPU)
!nvidia-smi

## Bước 2: Kết Nối Google Drive & Thiết Lập Thư Mục Sao Lưu Tự Động

In [ ]:
from google.colab import drive
import os

# Mount Google Drive vào môi trường Colab
drive.mount('/content/drive')

# Thiết lập thư mục lưu trữ an toàn trên Google Drive của bạn
DRIVE_BACKUP_DIR = '/content/drive/MyDrive/BattlefieldBlueprint_AI'
os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)
print(f"✅ Đã kết nối Google Drive! Thư mục sao lưu: {DRIVE_BACKUP_DIR}")

## Bước 3: Cài Đặt Trình Biên Dịch Rust & Kiểm Tra Môi Trường

In [ ]:
# Cài đặt Rust toolchain (cargo, rustc)
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y

import os
os.environ["PATH"] += ":/root/.cargo/bin"

!rustc --version
!cargo --version
!nvcc --version

## Bước 4: Tự Động Nạp Model Đã Có Từ Google Drive (Nếu có)

In [ ]:
import shutil

drive_model_file = os.path.join(DRIVE_BACKUP_DIR, "rl_model.json")
if os.path.exists(drive_model_file):
    print(f"🔄 Phát hiện model checkpoint trên Google Drive: {drive_model_file}")
    os.makedirs("assets", exist_ok=True)
    os.makedirs("rust-engine/assets", exist_ok=True)
    shutil.copy(drive_model_file, "assets/rl_model.json")
    shutil.copy(drive_model_file, "rust-engine/assets/rl_model.json")
    print("✅ Đã khôi phục model thành công vào workspace!")
else:
    print("🌱 Chưa có model checkpoint trên Google Drive. Sẽ bắt đầu huấn luyện từ model sẵn có trong repo hoặc khởi tạo mới.")

## Bước 5: Chạy Huấn Luyện AI Bot Đa Kịch Bản Trên GPU

- **EPISODES**: Số ván đấu cho mỗi kịch bản (ví dụ: 128 hoặc 256 ván).
- **DT**: Bước nhảy mô phỏng tua nhanh (0.25s).
- **MAX_TIME**: Thời gian tối đa mỗi ván (1800s = 30 phút game).
- **THREADS**: Số luồng CPU mô phỏng song song.
- **RESUME**: `true` để nạp model tiếp tục, `false` để huấn luyện từ đầu (Scratch).

In [ ]:
EPISODES = 128
DT = 0.25
MAX_TIME = 1800
THREADS = 4
RESUME = "true"

!chmod +x train_bot.sh
!./train_bot.sh {EPISODES} {DT} {MAX_TIME} {THREADS} {RESUME} {DRIVE_BACKUP_DIR}

## Bước 6: Kiểm Tra File Model & Checkpoints Đã Lưu Trên Google Drive

In [ ]:
import glob
import json

print("📂 Danh sách model & checkpoint trên Google Drive:")
files = glob.glob(f"{DRIVE_BACKUP_DIR}/*.json")
for f in sorted(files):
    size_kb = os.path.getsize(f) / 1024
    print(f"  - {os.path.basename(f)} ({size_kb:.1f} KB)")

main_model = os.path.join(DRIVE_BACKUP_DIR, "rl_model.json")
if os.path.exists(main_model):
    with open(main_model, "r") as mf:
        meta = json.load(mf)
    print("\n🧠 Thông tin Model AI Hợp Nhất:")
    print(f"   - Epsilon (Exploration): {meta.get('epsilon', 0.0):.4f}")
    print(f"   - Learning Rate:        {meta.get('learning_rate', 0.0):.5f}")
    print(f"   - Training Episodes:    {meta.get('training_episodes', 0)}")
    print(f"   - Total Parameters:     170,005 weights & biases")